# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description[:120]}...")

## 2. Data Overview
Review available record sets, fields, and their IDs.

All entities referenced by their `@id` as required.

In [ ]:
# List the available record sets by their @id
record_sets = []
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    # recordSet could be a list or dict-like object
    if isinstance(metadata.recordSet, list):
        for rs in metadata.recordSet:
            record_sets.append(rs['@id'] if isinstance(rs, dict) and '@id' in rs else str(rs))
    else:
        # single object
        record_sets.append(metadata.recordSet['@id'] if '@id' in metadata.recordSet else str(metadata.recordSet))
else:
    # In case recordSet is empty, try loading record sets from dataset directly
    try:
        record_sets = [x['@id'] for x in dataset.metadata.to_json()['recordSet']]
    except Exception as e:
        print("Could not find record sets.")

print("Available Record Sets (@id):")
for rs_id in record_sets:
    print(rs_id)

# For each record set, show sample records and field @ids
for rs_id in record_sets:
    print(f"\nSample records from record set {rs_id}:")
    records = list(dataset.records(record_set=rs_id))
    print(records[:2])
    # print field @ids
    try:
        rs_metadata = [x for x in dataset.metadata.to_json()['recordSet'] if x['@id'] == rs_id][0]
        fields = rs_metadata['field']
        if isinstance(fields, list):
            field_ids = [f['@id'] if isinstance(f, dict) else f for f in fields]
        else:
            field_ids = [fields['@id'] if isinstance(fields, dict) else fields]
        print("Fields (@id):", field_ids)
    except Exception as e:
        print('Could not retrieve fields.')

## 3. Data Extraction
Load data from all available record sets into DataFrames for analysis. Use the record set and field `@id`s found above.

In [ ]:
# Extract data from each record set
# Ensure record_sets is populated from previous section
if not record_sets:
    record_sets = [x['@id'] for x in dataset.metadata.to_json().get('recordSet', [])]

dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Record set {record_set_id} loaded: {df.shape[0]} records. Columns:")
    print(df.columns.tolist())

# Example: Show first 5 records from the first record set
if record_sets:
    first_rs_id = record_sets[0]
    print(f"\nFirst 5 records for record set {first_rs_id}:")
    print(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Below, we filter records with a numeric field (example: GAD-7 total score), normalize it, and group by a demographic field.

**All fields are referenced by their `@id`.**

In [ ]:
# Choose the record set to analyze (use the first one loaded)
record_set_id = record_sets[0]

# Inspect the field @ids to select numeric fields and group fields
df = dataframes[record_set_id]
print(f"Columns (@id)s for {record_set_id}:", df.columns.tolist())

# Example: Find GAD-7 score field and demographic field by their @id
numeric_field_id = None
group_field_id = None

# Pick commonly named fields (here by demonstration, you would use your actual field @id names)
# For this dataset, typical fields will be e.g. 'GAD7_total_score', 'age', 'level_of_education', etc.
for col in df.columns:
    if 'GAD7' in col or 'gad7' in col:
        numeric_field_id = col
    if 'level_of_education' in col or 'education_level' in col:
        group_field_id = col
    elif 'gender' in col or 'sex' in col:
        group_field_id = col

if numeric_field_id is None:
    # fallback to a numeric column
    numeric_candidates = [col for col in df.columns if df[col].dtype in [int, float]]
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]

if group_field_id is None:
    # fallback to any categorical column
    cat_candidates = [col for col in df.columns if df[col].dtype == 'object']
    if cat_candidates:
        group_field_id = cat_candidates[0]

print(f"Selected numeric field (@id): {numeric_field_id}")
print(f"Selected group field (@id): {group_field_id}")

# Filtering: Show records with numeric_field > threshold
threshold = 10
if numeric_field_id:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here, plot the numeric field distribution and/or group mean.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of numeric field
if numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

# Plot group means if grouping exists
if group_field_id and numeric_field_id and group_field_id in df.columns:
    group_means = df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    plt.figure(figsize=(10,5))
    sns.barplot(data=group_means, x=group_field_id, y=numeric_field_id)
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xlabel(group_field_id)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides rich survey data on mental health from Kilifi County, Kenya.
- Numeric fields such as assessment scores (e.g., GAD-7) can be analyzed for population distribution and subgroup differences.
- Demographic grouping (e.g., by education or gender) highlights potential variation and informs further research.
- Using `mlcroissant`, metadata and records are managed transparently, with all entities referenced by their `@id` to ensure traceability and reproducibility.